# start_here_rust: stage-by-stage pySTAMPS with the Rust/native backend

This notebook mirrors the teaching style of `02_pystamps_stage_execution.ipynb`, but the focus is the Rust/native backend.

Read it as a stage-by-stage guide. Each stage explains the goal, inputs, transformation, outputs, and backend role. Where the stage has a native Rust kernel, the notebook shows a Python-vs-Rust benchmark and numerical parity check. Where the stage has no Rust kernel, the notebook says so directly and keeps the stage in the workflow map.

The key distinction is simple: Rust is not a separate StaMPS workflow. Rust/native code accelerates selected kernels inside the same pySTAMPS stage contract.


## Legacy StaMPS map with Rust overlay

Legacy StaMPS groups several operations into broad MATLAB calls. pySTAMPS exposes those operations as stages, and the Rust backend accelerates selected kernels inside those stages.

| Legacy area | pySTAMPS stage | Rust/native role |
| --- | --- | --- |
| `stamps(1,4)` setup | Stage 1 | Python orchestration |
| `stamps(1,4)` gamma/coherence | Stage 2 | Rust grid, histogram, and topofit exports |
| `stamps(1,4)` selection | Stage 3 | Python orchestration consuming Stage-2 products |
| `stamps(1,4)` weeding | Stage 4 | Rust edge-statistics kernel |
| post merge | Stage 5 | Python orchestration |
| post unwrapping | Stage 6 | Python/external unwrap orchestration |
| post SCLA/velocity | Stage 7 | Rust SCLA kernel |
| final filtering | Stage 8 | Rust edge-noise kernel |


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import statistics
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from pystamps.kernels import (
    describe_backend_matrix,
    run_stage2_grid_accumulate_kernel,
    run_stage2_histogram_kernel,
    run_stage2_topofit_coh_row_invariant_kernel,
    run_stage2_topofit_kernel,
    run_stage2_topofit_row_invariant_kernel,
    run_stage4_edge_stats_kernel,
    run_stage7_scla_kernel,
    run_stage8_edge_noise_kernel,
)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'pystamps').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from this notebook')


REPO_ROOT = find_repo_root()
REFERENCE_ROOT = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag_hl'
FOCUSED_AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_small_baseline_audit.json'
RUST_BACKEND_EXPLAINER = REPO_ROOT / 'notebooks' / 'backend_rust_explanation.ipynb'
EVIDENCE_NOTEBOOK = REPO_ROOT / 'notebooks' / '04_rust_end2end_parity_validation.ipynb'

BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}

native_import_error = None
native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
try:
    native_mod = importlib.import_module('pystamps.kernels._stage2_native') if native_spec is not None else None
except Exception as exc:
    native_mod = None
    native_import_error = f'{type(exc).__name__}: {exc}'

native_path = Path(native_spec.origin) if native_spec is not None and native_spec.origin and native_mod is not None else None


def relpath(path: str | Path | None) -> str:
    if path is None:
        return '<missing>'
    candidate = Path(path)
    try:
        return str(candidate.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(candidate)


def modified_utc(path: Path | None) -> str:
    if path is None or not path.exists():
        return '<missing>'
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat(timespec='seconds')


context_df = pd.DataFrame(
    [
        {'item': 'repo root', 'value': relpath(REPO_ROOT)},
        {'item': 'teaching reference dataset', 'value': relpath(REFERENCE_ROOT)},
        {'item': 'focused StaMPS audit artifact', 'value': relpath(FOCUSED_AUDIT_JSON)},
        {'item': 'compiled Rust/native module', 'value': relpath(native_path) if native_path else '<not importable>'},
        {'item': 'native module modified UTC', 'value': modified_utc(native_path)},
        {'item': 'native import error', 'value': native_import_error or '<none>'},
    ]
)
display(context_df)


,item,value
0,repo root,.
1,teaching reference dataset,inputs_and_outputs/InSAR_dataset_test_stage8di...
2,focused StaMPS audit artifact,inputs_and_outputs/validation_runs/latest_smal...
3,compiled Rust/native module,pystamps/kernels/_stage2_native.cpython-314-x8...
4,native module modified UTC,2026-04-28T07:40:10+00:00
5,native import error,<none>


## Prepare the Rust/backend context

Notebook `02` prepares a scratch dataset before running or replaying stages. This Rust notebook prepares a backend context instead: the compiled native module, the backend matrix, and the focused StaMPS parity artifact used later.

Use this runtime shape when you want to force native kernels in a real run:

```yaml
runtime:
  backend: native
  stage2_kernel_backend: native
  stage2_native_threads: 8
```


In [2]:
backend_matrix = describe_backend_matrix()
speed_claim_kernels = {
    'stage2_grid_accumulate',
    'stage2_histogram',
    'stage2_topofit',
    'stage2_topofit_coh_row_invariant',
    'stage2_topofit_row_invariant',
    'stage4_edge_stats',
    'stage7_scla',
    'stage8_edge_noise',
}
backend_rows = []
for kernel, info in sorted(backend_matrix.get('kernels', {}).items()):
    available = info.get('available_backends', [])
    backend_rows.append(
        {
            'kernel': kernel,
            'python_reference': info.get('baseline_backend', 'python'),
            'native_available_now': 'native' in available,
            'available_backends': ', '.join(available),
            'rust_speed_claim': 'yes' if kernel in speed_claim_kernels and 'native' in available else 'no',
            'stage': '2' if kernel.startswith('stage2_') else kernel.replace('stage', '').split('_', 1)[0],
            'reader_note': 'measured below against Python' if kernel in speed_claim_kernels and 'native' in available else 'not measured here',
        }
    )
backend_df = pd.DataFrame(backend_rows)
display(backend_df)


,kernel,python_reference,native_available_now,available_backends,rust_speed_claim,stage,reader_note
0,stage2_grid_accumulate,python,True,"native, python",yes,2,measured below against Python
1,stage2_histogram,python,True,"native, python",yes,2,measured below against Python
2,stage2_topofit,python,True,"native, python",yes,2,measured below against Python
3,stage2_topofit_coh_row_invariant,python,True,"native, python",yes,2,measured below against Python
4,stage2_topofit_row_invariant,python,True,"native, python",yes,2,measured below against Python
5,stage4_edge_stats,python,True,"native, python",yes,4,measured below against Python
6,stage7_scla,python,True,"cuda, native, python",yes,7,measured below against Python
7,stage8_edge_noise,python,True,"cuda, native, python",yes,8,measured below against Python


## Helper functions and benchmark harness

Notebook `02` uses helpers to run one stage, verify one stage, and render one stage report. This notebook uses the same idea, but the helper output is backend-specific:

- a stage report says what the stage does,
- a Rust role says whether native code participates,
- a benchmark table appears when a native kernel exists,
- a parity column must be true before a speedup is meaningful.


In [3]:
bench_rows: list[dict[str, object]] = []


def timed(fn: Callable[[], Any], repeats: int) -> tuple[list[float], Any]:
    fn()
    runs: list[float] = []
    output: Any = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        output = fn()
        runs.append(time.perf_counter() - t0)
    return runs, output


def output_items(value: Any) -> dict[str, np.ndarray]:
    if isinstance(value, dict):
        return {str(key): np.asarray(item) for key, item in value.items()}
    if isinstance(value, tuple):
        return {f'out_{idx}': np.asarray(item) for idx, item in enumerate(value)}
    return {'out': np.asarray(value)}


def max_abs_array(left: np.ndarray, right: np.ndarray) -> float:
    left_arr = np.asarray(left)
    right_arr = np.asarray(right)
    if left_arr.shape != right_arr.shape:
        return float('inf')
    if np.array_equal(left_arr, right_arr, equal_nan=True):
        return 0.0
    finite = np.isfinite(left_arr) & np.isfinite(right_arr)
    if not np.array_equal(left_arr[~finite], right_arr[~finite], equal_nan=True):
        return float('inf')
    return float(np.max(np.abs(left_arr[finite] - right_arr[finite]))) if np.any(finite) else 0.0


def max_abs_output(left: Any, right: Any) -> float:
    left_items = output_items(left)
    right_items = output_items(right)
    if set(left_items) != set(right_items):
        return float('inf')
    return max(max_abs_array(left_items[key], right_items[key]) for key in sorted(left_items))


def native_topofit_generic(cpxphase: np.ndarray, bperp: np.ndarray, n_trial_wraps: float, threads: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    result = native_mod.ps_topofit_batch_generic(
        np.ascontiguousarray(cpxphase, dtype=np.complex128),
        np.ascontiguousarray(bperp, dtype=np.float64),
        float(n_trial_wraps),
        int(threads),
    )
    return tuple(np.asarray(item) for item in result)


def native_topofit_row(cpxphase: np.ndarray, bperp_vec: np.ndarray, n_trial_wraps: float, threads: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    result = native_mod.ps_topofit_batch_row_invariant(
        np.ascontiguousarray(cpxphase, dtype=np.complex128),
        np.ascontiguousarray(bperp_vec, dtype=np.float64),
        float(n_trial_wraps),
        int(threads),
    )
    return tuple(np.asarray(item) for item in result)


def native_topofit_row_coh(cpxphase: np.ndarray, bperp_vec: np.ndarray, n_trial_wraps: float, threads: int) -> np.ndarray:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    return np.asarray(
        native_mod.ps_topofit_coh_row_invariant(
            np.ascontiguousarray(cpxphase, dtype=np.complex128),
            np.ascontiguousarray(bperp_vec, dtype=np.float64),
            float(n_trial_wraps),
            int(threads),
        ),
        dtype=np.float64,
    )


def record_kernel(kernel: str, python_fn: Callable[[], Any], native_fn: Callable[[], Any], tolerance: float, repeats: int) -> None:
    python_runs, python_output = timed(python_fn, repeats)
    native_runs, native_output = timed(native_fn, repeats)
    python_mean = statistics.mean(python_runs)
    native_mean = statistics.mean(native_runs)
    max_abs = max_abs_output(python_output, native_output)
    bench_rows.append(
        {
            'kernel': kernel,
            'python_mean_sec': python_mean,
            'native_mean_sec': native_mean,
            'speedup_vs_python': python_mean / native_mean if native_mean > 0 else float('inf'),
            'max_abs': max_abs,
            'tolerance': tolerance,
            'parity_ok': bool(max_abs <= tolerance),
            'rust_faster': bool(native_mean < python_mean),
            'python_runs_sec': ', '.join(f'{value:.4f}' for value in python_runs),
            'native_runs_sec': ', '.join(f'{value:.4f}' for value in native_runs),
        }
    )


if native_mod is None:
    bench_rows.append({'kernel': '<native unavailable>', 'parity_ok': False, 'rust_faster': False})
else:
    repeats = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '1'))
    native_threads = int(os.environ.get('PYSTAMPS_NOTEBOOK_NATIVE_THREADS', '8'))

    rng_grid = np.random.default_rng(101)
    n_grid_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_PS', '100000'))
    n_grid_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_IFG', '24'))
    n_grid_i = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_I', '500'))
    n_grid_j = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_J', '20'))
    ph_weight = np.exp(1j * rng_grid.normal(size=(n_grid_ps, n_grid_ifg))).astype(np.complex64)
    grid_lin = rng_grid.integers(0, n_grid_i * n_grid_j, size=n_grid_ps, dtype=np.int64)
    record_kernel('stage2_grid_accumulate', lambda: run_stage2_grid_accumulate_kernel(ph_weight, grid_lin, n_grid_i, n_grid_j, backend='python'), lambda: run_stage2_grid_accumulate_kernel(ph_weight, grid_lin, n_grid_i, n_grid_j, backend='native', threads=native_threads), 1.0e-6, repeats)

    rng_hist = np.random.default_rng(102)
    hist_values = rng_hist.normal(size=int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_HIST_VALUES', '2000000'))).astype(np.float64)
    hist_centers = np.linspace(-5.0, 5.0, int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_HIST_CENTERS', '401')), dtype=np.float64)
    record_kernel('stage2_histogram', lambda: run_stage2_histogram_kernel(hist_values, hist_centers, backend='python'), lambda: run_stage2_histogram_kernel(hist_values, hist_centers, backend='native'), 0.0, repeats)

    rng_topo = np.random.default_rng(103)
    n_topo_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_TOPOFIT_PS', '3000'))
    n_topo_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_TOPOFIT_IFG', '28'))
    cpx_topo = np.exp(1j * rng_topo.normal(size=(n_topo_ps, n_topo_ifg))).astype(np.complex128)
    bperp_topo = rng_topo.normal(size=(n_topo_ps, n_topo_ifg)).astype(np.float64)
    record_kernel('stage2_topofit', lambda: run_stage2_topofit_kernel(cpx_topo, bperp_topo, 1.5, backend='python'), lambda: native_topofit_generic(cpx_topo, bperp_topo, 1.5, native_threads), 1.0e-9, repeats)

    rng_topo_row = np.random.default_rng(99)
    cpx_row = np.exp(1j * rng_topo_row.normal(size=(n_topo_ps, n_topo_ifg))).astype(np.complex128)
    bperp_row = np.linspace(-120.0, 120.0, n_topo_ifg, dtype=np.float64)
    bperp_row_mat = np.broadcast_to(bperp_row, (n_topo_ps, n_topo_ifg)).copy()
    record_kernel('stage2_topofit_row_invariant', lambda: run_stage2_topofit_row_invariant_kernel(cpx_row, bperp_row_mat, 1.5, backend='python'), lambda: native_topofit_row(cpx_row, bperp_row, 1.5, native_threads), 1.0e-9, repeats)

    rng_topo_coh = np.random.default_rng(104)
    n_coh_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_COH_PS', '5000'))
    n_coh_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_COH_IFG', '80'))
    cpx_coh = np.exp(1j * rng_topo_coh.normal(size=(n_coh_ps, n_coh_ifg))).astype(np.complex128)
    bperp_coh = np.linspace(-120.0, 120.0, n_coh_ifg, dtype=np.float64)
    bperp_coh_mat = np.broadcast_to(bperp_coh, (n_coh_ps, n_coh_ifg)).copy()
    record_kernel('stage2_topofit_coh_row_invariant', lambda: run_stage2_topofit_coh_row_invariant_kernel(cpx_coh, bperp_coh_mat, 1.5, backend='python'), lambda: native_topofit_row_coh(cpx_coh, bperp_coh, 1.5, native_threads), 1.0e-9, repeats)

    rng4 = np.random.default_rng(321)
    n_ps4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_PS', '30000'))
    n_ifg4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_IFG', '32'))
    n_edge4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_EDGES', '60000'))
    ph_weed = np.exp(1j * rng4.normal(size=(n_ps4, n_ifg4))).astype(np.complex128)
    node_a4 = rng4.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    node_b4 = rng4.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    bperp4 = np.linspace(-80.0, 120.0, n_ifg4, dtype=np.float64)
    day4 = np.linspace(0.0, 365.0, n_ifg4, dtype=np.float64)
    record_kernel('stage4_edge_stats', lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='python'), lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='native'), 1.0e-10, repeats)

    rng7 = np.random.default_rng(322)
    n_ps7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_PS', '30000'))
    n_ifg7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_IFG', '36'))
    ph_proc = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    ph_mean_v = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    bperp_mat = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    unwrap_ix = np.arange(n_ifg7, dtype=np.int64)
    solve_ix = np.arange(n_ifg7, dtype=np.int64)
    day7 = np.linspace(0.0, 365.0, n_ifg7, dtype=np.float64)
    ifg_std = np.full(n_ifg7, 10.0, dtype=np.float64)
    record_kernel('stage7_scla', lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='python'), lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='native'), 1.0e-4, repeats)

    rng8 = np.random.default_rng(323)
    n_ps8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_PS', '50000'))
    n_ifg8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_IFG', '36'))
    n_edge8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_EDGES', '120000'))
    uw_ph = np.exp(1j * rng8.normal(size=(n_ps8, n_ifg8))).astype(np.complex64)
    node_a8 = rng8.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    node_b8 = rng8.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    record_kernel('stage8_edge_noise', lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='python'), lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='native'), 1.0e-5, repeats)

bench_df = pd.DataFrame(bench_rows)
bench_display = bench_df.copy()
for column in ['python_mean_sec', 'native_mean_sec', 'speedup_vs_python']:
    if column in bench_display:
        bench_display[column] = bench_display[column].map(lambda value: round(float(value), 4) if pd.notna(value) else value)
for column in ['max_abs', 'tolerance']:
    if column in bench_display:
        bench_display[column] = bench_display[column].map(lambda value: f'{float(value):.2e}' if pd.notna(value) else value)

speed_ok = bool(not bench_df.empty and bench_df.get('parity_ok', pd.Series(dtype=bool)).all() and bench_df.get('rust_faster', pd.Series(dtype=bool)).all())
min_speedup = round(float(bench_df['speedup_vs_python'].min()), 3) if not bench_df.empty and 'speedup_vs_python' in bench_df else '<not measured>'

display(bench_display[['kernel', 'python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance', 'parity_ok', 'rust_faster']])
display(pd.DataFrame([
    {'check': 'all measured Rust/native kernels match Python numerically', 'result': bool(bench_df.get('parity_ok', pd.Series(dtype=bool)).all())},
    {'check': 'all measured Rust/native kernels are faster', 'result': bool(bench_df.get('rust_faster', pd.Series(dtype=bool)).all())},
    {'check': 'minimum native speedup versus Python', 'result': min_speedup},
]))


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster
0,stage2_grid_accumulate,0.0193,0.0051,3.7915,0.00e+00,1.00e-06,True,True
1,stage2_histogram,0.1386,0.1079,1.2843,0.00e+00,0.00e+00,True,True
2,stage2_topofit,0.4589,0.0133,34.6266,5.55e-16,1.00e-09,True,True
3,stage2_topofit_row_invariant,0.4571,0.0034,135.8809,4.44e-16,1.00e-09,True,True
4,stage2_topofit_coh_row_invariant,0.0571,0.0104,5.4827,6.66e-16,1.00e-09,True,True
5,stage4_edge_stats,5.1270,0.7083,7.2384,2.22e-15,1.00e-10,True,True
6,stage7_scla,0.0753,0.0155,4.8473,3.05e-05,1.00e-04,True,True
7,stage8_edge_noise,0.0995,0.0416,2.3919,4.77e-07,1.00e-05,True,True


,check,result
0,all measured Rust/native kernels match Python ...,True
1,all measured Rust/native kernels are faster,True
2,minimum native speedup versus Python,1.284


In [4]:
STAGE_ROWS = {
    1: {
        'stage': '1. Initial load',
        'rust_backend_role': 'Python orchestration; no Rust kernel is used in this stage.',
        'input': 'raw candidate coordinates, phase samples, dates, baselines, and metadata',
        'operator': 'normalize candidate geometry and insert the master acquisition',
        'output': 'patch-local ps1/ph1/bp1/da1/hgt1 state',
        'validation': 'stage-1 artifacts are compared to the StaMPS reference in notebook 02',
    },
    2: {
        'stage': '2. Estimate gamma/coherence',
        'rust_backend_role': 'Rust exports accelerate grid accumulation, histogramming, and topofit kernels.',
        'input': 'ph1, bp1, ps1, candidate weights, trial-wrap parameters',
        'operator': 'fit K/C/coherence, accumulate weighted phase, and calibrate random coherence',
        'output': 'pm1 with K_ps, C_ps, coh_ps, ph_res, ph_grid, ph_patch, Nr',
        'validation': 'native kernels below match Python reference outputs within tolerance',
    },
    3: {
        'stage': '3. Select persistent scatterers',
        'rust_backend_role': 'Python orchestration consumes Stage-2 outputs produced under the native backend contract.',
        'input': 'Stage-2 coherence and filtering diagnostics',
        'operator': 'threshold and retain stable candidate points',
        'output': 'select1.mat retained-candidate index mapping',
        'validation': 'selection artifacts are stage-scoped in notebook 02',
    },
    4: {
        'stage': '4. Weed adjacent/noisy candidates',
        'rust_backend_role': 'Rust `stage4_edge_stats` accelerates repeated edge statistics.',
        'input': 'selected candidate set, neighborhood edges, phase/baseline vectors',
        'operator': 'remove spatially redundant or locally unstable candidates',
        'output': 'weed1.mat masks and survivor index vectors',
        'validation': 'native edge statistics match Python reference outputs below',
    },
    5: {
        'stage': '5. Promote patch outputs and merge',
        'rust_backend_role': 'Python orchestration; no Rust kernel is used in this merge step.',
        'input': 'patch-local survivors from stages 1-4',
        'operator': 'concatenate and align patch-level products into root-level products',
        'output': 'merged ps2/ph2/pm2/bp2/hgt2/la2/rc2 artifacts',
        'validation': 'merged artifacts are stage-scoped in notebook 02',
    },
    6: {
        'stage': '6. Unwrap phase products',
        'rust_backend_role': 'Python orchestration and external unwrap tooling; no Rust kernel is used in this stage.',
        'input': 'merged wrapped phase products and graph/grid support',
        'operator': 'resolve 2*pi ambiguity into unwrapped phase products',
        'output': 'phuw2, uw_phaseuw, uw_grid, uw_interp, ifgstd2',
        'validation': 'unwrap artifacts are stage-scoped in notebook 02',
    },
    7: {
        'stage': '7. Estimate SCLA and velocity',
        'rust_backend_role': 'Rust `stage7_scla` accelerates the SCLA/velocity regression kernel.',
        'input': 'unwrapped phase, baselines, acquisition time, solve indices',
        'operator': 'fit spatially correlated look-angle error and temporal velocity terms',
        'output': 'scla2, scla_smooth2, mean_v, mv2',
        'validation': 'focused small-baseline StaMPS audit rows are shown later in this notebook',
    },
    8: {
        'stage': '8. Final space-time filtering',
        'rust_backend_role': 'Rust `stage8_edge_noise` accelerates edge-noise calculations.',
        'input': 'unwrapped phase products and edge graph',
        'operator': 'separate residual spatial/temporal noise from structured phase',
        'output': 'uw_space_time.mat final filtered product',
        'validation': 'native edge-noise output matches Python reference outputs below',
    },
}

STAGE_KERNELS = {
    2: ['stage2_grid_accumulate', 'stage2_histogram', 'stage2_topofit', 'stage2_topofit_row_invariant', 'stage2_topofit_coh_row_invariant'],
    4: ['stage4_edge_stats'],
    7: ['stage7_scla'],
    8: ['stage8_edge_noise'],
}


def show_stage(stage_id: int) -> dict[str, object]:
    row = STAGE_ROWS[stage_id]
    display(pd.DataFrame([row]))
    kernels = STAGE_KERNELS.get(stage_id, [])
    if kernels:
        cols = ['kernel', 'python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance', 'parity_ok', 'rust_faster']
        stage_bench = bench_display[bench_display['kernel'].isin(kernels)][cols]
        display(stage_bench)
        return {'stage': row, 'kernels': kernels, 'speed_ok': bool(stage_bench['rust_faster'].all() and stage_bench['parity_ok'].all())}
    display(pd.DataFrame([{'kernel': '<none>', 'reason': 'This stage has no registered Rust/native kernel; it is still part of the native-backed workflow context.'}]))
    return {'stage': row, 'kernels': [], 'speed_ok': None}


## Notation and backend model

We keep the same scientific notation as notebook `02`:

- `p`: persistent-scatterer candidate index
- `k`: interferogram index
- `z[p,k] = exp(j * phi[p,k])`: complex phase sample
- `b_perp[p,k]`: perpendicular baseline
- `K[p]`, `C[p]`, `gamma[p]`: Stage-2 topofit outputs

The Rust/backend notation adds one implementation choice:

```text
A_stage = Python orchestration + selected native kernel calls
```

The contract remains unchanged: native outputs must match the Python reference outputs within the shown tolerance.


## How to read the stage tables

Each stage has one report cell. For stages with Rust kernels, the second table is the benchmark evidence:

- `parity_ok` must be true.
- `rust_faster` must be true.
- `speedup_vs_python` is the observed speedup for this executed notebook run.

For stages without Rust kernels, the report says `<none>` for measured kernels. That is not a failure; it means the stage is orchestration or I/O rather than a native numerical hotspot.


## Stage 1. Initial load

Stage 1 converts raw candidate lists and campaign metadata into the normalized patch state used downstream. It defines indexing, geometry, dates, baselines, and master-acquisition insertion.

Rust role: no Rust kernel is registered for Stage 1. The native backend context starts to matter once later numerical kernels are reached.


In [5]:
stage_1 = show_stage(1)


,stage,rust_backend_role,input,operator,output,validation
0,1. Initial load,Python orchestration; no Rust kernel is used i...,"raw candidate coordinates, phase samples, date...",normalize candidate geometry and insert the ma...,patch-local ps1/ph1/bp1/da1/hgt1 state,stage-1 artifacts are compared to the StaMPS r...


,kernel,reason
0,<none>,This stage has no registered Rust/native kerne...


## Stage 2. Estimate gamma / coherence

Stage 2 estimates residual topographic phase slope, constant phase offset, and coherence-like stability. This is the first Rust-heavy stage.

Rust role: the notebook benchmarks grid accumulation, histogramming, generic topofit, row-invariant topofit, and row-invariant coherence topofit against Python reference outputs.


In [6]:
stage_2 = show_stage(2)


,stage,rust_backend_role,input,operator,output,validation
0,2. Estimate gamma/coherence,"Rust exports accelerate grid accumulation, his...","ph1, bp1, ps1, candidate weights, trial-wrap p...","fit K/C/coherence, accumulate weighted phase, ...","pm1 with K_ps, C_ps, coh_ps, ph_res, ph_grid, ...",native kernels below match Python reference ou...


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster
0,stage2_grid_accumulate,0.0193,0.0051,3.7915,0.00e+00,1.00e-06,True,True
1,stage2_histogram,0.1386,0.1079,1.2843,0.00e+00,0.00e+00,True,True
2,stage2_topofit,0.4589,0.0133,34.6266,5.55e-16,1.00e-09,True,True
3,stage2_topofit_row_invariant,0.4571,0.0034,135.8809,4.44e-16,1.00e-09,True,True
4,stage2_topofit_coh_row_invariant,0.0571,0.0104,5.4827,6.66e-16,1.00e-09,True,True


## Stage 3. Select persistent scatterers

Stage 3 reduces the candidate set using Stage-2 diagnostics. In set notation, it chooses a subset `S3` from the candidate set `S2`.

Rust role: no separate Stage-3 Rust kernel is registered. The stage consumes the products produced by the Stage-2 backend contract.


In [7]:
stage_3 = show_stage(3)


,stage,rust_backend_role,input,operator,output,validation
0,3. Select persistent scatterers,Python orchestration consumes Stage-2 outputs ...,Stage-2 coherence and filtering diagnostics,threshold and retain stable candidate points,select1.mat retained-candidate index mapping,selection artifacts are stage-scoped in notebo...


,kernel,reason
0,<none>,This stage has no registered Rust/native kerne...


## Stage 4. Weed adjacent / noisy candidates

Stage 4 removes redundant or unstable candidates using neighborhood relationships and edge statistics.

Rust role: `stage4_edge_stats` accelerates the repeated edge-statistics calculation and is measured below.


In [8]:
stage_4 = show_stage(4)


,stage,rust_backend_role,input,operator,output,validation
0,4. Weed adjacent/noisy candidates,Rust `stage4_edge_stats` accelerates repeated ...,"selected candidate set, neighborhood edges, ph...",remove spatially redundant or locally unstable...,weed1.mat masks and survivor index vectors,native edge statistics match Python reference ...


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster
5,stage4_edge_stats,5.127,0.7083,7.2384,2.22e-15,1.00e-10,True,True


## Stage 5. Promote patch outputs and merge

Stage 5 changes representation from patch-local products to merged campaign-level artifacts.

Rust role: no Rust kernel is registered for this merge step. It is bookkeeping and artifact promotion.


In [9]:
stage_5 = show_stage(5)


,stage,rust_backend_role,input,operator,output,validation
0,5. Promote patch outputs and merge,Python orchestration; no Rust kernel is used i...,patch-local survivors from stages 1-4,concatenate and align patch-level products int...,merged ps2/ph2/pm2/bp2/hgt2/la2/rc2 artifacts,merged artifacts are stage-scoped in notebook 02


,kernel,reason
0,<none>,This stage has no registered Rust/native kerne...


## Stage 6. Unwrap phase products

Stage 6 resolves the `2*pi` ambiguity and produces unwrapped phase products for later geophysical fitting.

Rust role: no Rust kernel is registered for Stage 6 in this backend matrix. The stage remains part of the native-backed workflow context, but the numerical hotspot is not a Rust export here.


In [10]:
stage_6 = show_stage(6)


,stage,rust_backend_role,input,operator,output,validation
0,6. Unwrap phase products,Python orchestration and external unwrap tooli...,merged wrapped phase products and graph/grid s...,resolve 2*pi ambiguity into unwrapped phase pr...,"phuw2, uw_phaseuw, uw_grid, uw_interp, ifgstd2",unwrap artifacts are stage-scoped in notebook 02


,kernel,reason
0,<none>,This stage has no registered Rust/native kerne...


## Stage 7. Estimate SCLA and velocity products

Stage 7 fits spatially correlated look-angle error and velocity terms from unwrapped phase products.

Rust role: `stage7_scla` accelerates the SCLA/velocity regression kernel. The focused StaMPS audit artifact later in this notebook is also Stage-7 small-baseline parity evidence.


In [11]:
stage_7 = show_stage(7)


,stage,rust_backend_role,input,operator,output,validation
0,7. Estimate SCLA and velocity,Rust `stage7_scla` accelerates the SCLA/veloci...,"unwrapped phase, baselines, acquisition time, ...",fit spatially correlated look-angle error and ...,"scla2, scla_smooth2, mean_v, mv2",focused small-baseline StaMPS audit rows are s...


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster
6,stage7_scla,0.0753,0.0155,4.8473,3.05e-05,1.00e-04,True,True


## Stage 8. Final space-time filtering

Stage 8 separates residual spatial/temporal noise from structured phase terms and writes the final filtered product.

Rust role: `stage8_edge_noise` accelerates edge-noise calculations and is measured below.


In [12]:
stage_8 = show_stage(8)


,stage,rust_backend_role,input,operator,output,validation
0,8. Final space-time filtering,Rust `stage8_edge_noise` accelerates edge-nois...,unwrapped phase products and edge graph,separate residual spatial/temporal noise from ...,uw_space_time.mat final filtered product,native edge-noise output matches Python refere...


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster
7,stage8_edge_noise,0.0995,0.0416,2.3919,4.77e-07,1.00e-05,True,True


## Rust parity diagnostics and StaMPS validation

Notebook `02` validates stage artifacts against the bundled StaMPS reference. This Rust notebook keeps the validation story focused: it reads the latest focused small-baseline audit artifact and confirms it completed cleanly.

This is not a full manifest claim. Use `make audit` for the full maintained parity gate.


In [13]:
if not FOCUSED_AUDIT_JSON.exists():
    raise FileNotFoundError(f'Missing focused audit artifact: {FOCUSED_AUDIT_JSON}')

audit_payload = json.loads(FOCUSED_AUDIT_JSON.read_text(encoding='utf-8'))
audit_rows = audit_payload.get('audits', [])
audit_ok = bool(audit_payload.get('ok') and audit_payload.get('completed') and not audit_payload.get('interrupted'))

audit_df = pd.DataFrame(
    [
        {
            'dataset': audit.get('dataset'),
            'workflow': audit.get('workflow'),
            'status': audit.get('status'),
            'ok': bool(audit.get('ok')),
            'checked_files': int(audit.get('checked', 0)),
            'failed_files': len(audit.get('failures', [])),
            'run_root': relpath(audit.get('run_root')),
            'golden_root': relpath(audit.get('golden_root')),
        }
        for audit in audit_rows
    ]
)

display(pd.DataFrame([
    {'check': 'focused audit artifact exists', 'result': FOCUSED_AUDIT_JSON.exists()},
    {'check': 'focused audit completed', 'result': bool(audit_payload.get('completed'))},
    {'check': 'focused audit interrupted', 'result': bool(audit_payload.get('interrupted'))},
    {'check': 'focused audit ok', 'result': audit_ok},
    {'check': 'full manifest claimed here', 'result': False},
]))
display(audit_df)


,check,result
0,focused audit artifact exists,True
1,focused audit completed,True
2,focused audit interrupted,False
3,focused audit ok,True
4,full manifest claimed here,False


,dataset,workflow,status,ok,checked_files,failed_files,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,InSAR_dataset_small_baseline_stage7diag_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...
1,InSAR_dataset_small_baseline_stage7,InSAR_dataset_small_baseline_stage7_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...


## Final checklist

This is the compact reading check for the notebook. If the first five rows are true, the notebook has shown the native module, all native-kernel speed claims, stage-by-stage backend roles, and focused StaMPS parity evidence.


In [14]:
stage_results = [stage_1, stage_2, stage_3, stage_4, stage_5, stage_6, stage_7, stage_8]
final_checks = pd.DataFrame(
    [
        {'question': 'Is the Rust/native module importable?', 'answer': native_mod is not None},
        {'question': 'Does every native kernel row have a Rust speed claim?', 'answer': bool((backend_df['rust_speed_claim'] == 'yes').all())},
        {'question': 'Do measured Rust/native kernels match Python?', 'answer': bool(bench_df['parity_ok'].all())},
        {'question': 'Are measured Rust/native kernels faster than Python?', 'answer': bool(bench_df['rust_faster'].all())},
        {'question': 'Did the focused StaMPS audit pass?', 'answer': audit_ok},
        {'question': 'Did this notebook claim full manifest parity?', 'answer': False},
    ]
)
display(final_checks)


,question,answer
0,Is the Rust/native module importable?,True
1,Does every native kernel row have a Rust speed...,True
2,Do measured Rust/native kernels match Python?,True
3,Are measured Rust/native kernels faster than P...,True
4,Did the focused StaMPS audit pass?,True
5,Did this notebook claim full manifest parity?,False


## Next step

- Open `backend_rust_explanation.ipynb` for the conceptual Rust/native backend explainer.
- Open `04_rust_end2end_parity_validation.ipynb` for the evidence appendix with explicit verify reruns and pytest smoke checks.
- Open `02_pystamps_stage_execution.ipynb` when you want the full original stage-execution walkthrough.
- Run `make audit` when you need the full maintained parity gate.
